# grpc_server

> Fill in a module description here

In [ ]:
# | default_exp grpc_server

In [ ]:
# | export
from typing import Any, Callable, Generator

import grpc

from spannerflow.dataflow.v1 import dataflow_pb2, dataflow_pb2_grpc

# class FunctionService

In [ ]:
# | export
class FunctionService(dataflow_pb2_grpc.FunctionServiceServicer):
    def __init__(
        self,
        ie_functions: dict[
            str, tuple[str, Callable[[Any], Any], list[type], list[type]]
        ],
        agg_functions: dict[
            str, tuple[str, Callable[[Any], Any], list[type], list[type]]
        ],
    ):
        super().__init__()
        self._ie_functions = ie_functions
        self._agg_functions = agg_functions

    def RunAggFunction(
        self,
        request_iterator: Generator[dataflow_pb2.RunAggFunctionRequest, None, None],  # type: ignore
        context,
    ) -> Generator[dataflow_pb2.RunAggFunctionResponse, None, None]:  # type: ignore
        function_name = None
        func = None
        rows = []
        for request in request_iterator:
            if request.HasField("function_name"):
                if function_name is not None:
                    context.set_details("Function name already provided.")
                    context.set_code(grpc.StatusCode.INVALID_ARGUMENT)
                    return
                # Extract the function_name from the first request
                function_name = request.function_name
                func_tuple = self._agg_functions.get(function_name)
                print(func_tuple)

                if func_tuple is None:
                    context.set_details(f"Agg Function '{function_name}' not found.")
                    context.set_code(grpc.StatusCode.NOT_FOUND)
                    return  # Return early after setting the error
                func = func_tuple[1]
                func_in_scehma = func_tuple[2]
                func_out_schema = func_tuple[3]

            elif request.HasField("row"):
                if func is None:
                    context.set_details("Function name must be provided before rows.")
                    context.set_code(grpc.StatusCode.INVALID_ARGUMENT)
                    return  # Function name must come first
                str_arguments = [str(cell) for cell in request.row.row]
                val = [func_in_scehma[i](cell) for i, cell in enumerate(str_arguments)]
                rows.append(val[0] if len(val) == 1 else val)

        if function_name is None or func is None:
            context.set_details("No function name provided.")
            context.set_code(grpc.StatusCode.INVALID_ARGUMENT)
            return

        print(rows)
        res = func(rows)
        print(res)

        if len(func_out_schema) > 1:
            response_row = [
                (str(cell) if func_out_schema[i] is not bool else str(cell).lower())
                for i, cell in enumerate(res)
            ]
        else:
            response_row = [str(res) if res is not bool else str(res).lower()]

        response = dataflow_pb2.RunIEFunctionResponse(row=response_row)  # type: ignore

        yield response

    def RunIEFunction(
        self,
        request_iterator: Generator[dataflow_pb2.RunIEFunctionRequest, None, None],  # type: ignore
        context,
    ) -> Generator[dataflow_pb2.RunIEFunctionResponse, None, None]:  # type: ignore
        function_name = None
        func = None
        rows = []
        for request in request_iterator:
            if request.HasField("function_name"):
                if function_name is not None:
                    context.set_details("Function name already provided.")
                    context.set_code(grpc.StatusCode.INVALID_ARGUMENT)
                    return
                # Extract the function_name from the first request
                function_name = request.function_name
                func_tuple = self._ie_functions.get(function_name)

                if func_tuple is None:
                    context.set_details(f"IE Function '{function_name}' not found.")
                    context.set_code(grpc.StatusCode.NOT_FOUND)
                    return  # Return early after setting the error
                func = func_tuple[1]
                func_in_scehma = func_tuple[2]
                func_out_schema = func_tuple[3]

            elif request.HasField("row"):
                if func is None:
                    context.set_details("Function name must be provided before rows.")
                    context.set_code(grpc.StatusCode.INVALID_ARGUMENT)
                    return  # Function name must come first
                str_arguments = [str(cell) for cell in request.row.row]
                rows.append(
                    [func_in_scehma[i](cell) for i, cell in enumerate(str_arguments)]
                )

        if function_name is None or func is None:
            context.set_details("No function name provided.")
            context.set_code(grpc.StatusCode.INVALID_ARGUMENT)
            return

        for row in rows:
            output = func(*row)
            if not isinstance(output, (tuple, list)):
                output = (output,)
            for r in output:
                response_row = [
                    (str(cell) if func_out_schema[i] is not bool else str(cell).lower())
                    for i, cell in enumerate(r)
                ]
                response = dataflow_pb2.RunIEFunctionResponse(  # type: ignore
                    row=response_row
                )
                yield response

In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()